# Modelling noisy circuits — a stochastic noise model on the sliding-window MPS decoder

*Directory 7 · builds directly on notebook `6.Real-Time-Decoder/example_sliding_window_itensor.ipynb`.*

So far every error in this series was **hand-injected**: we placed a specific `X`, `Z`, `Y` or
measurement fault at a chosen qubit and round to prove the sliding-window decoder catches it. That is
the right way to *verify* a decoder, but it is not how a real device fails. On hardware, errors happen
**everywhere, all the time, at random**, and the amount of error that accumulates depends on **how long
the qubits sit idle** — in particular on how many syndrome-extraction rounds separate one logical gate
from the next.

This notebook turns the hand-injection hooks into a genuine **stochastic noise model** and drives the
full stack with it:

* a per-round Pauli channel at rate `p` (X, Y and Z errors on the data qubits) and a measurement-flip
  channel at rate `q`, applied on **every** syndrome round;
* an explicit **timestep**: an epoch of `R` rounds is the unit of time between logical gates, so more
  rounds ⇒ more accumulated noise;
* the **sliding-window MWPM decoder with artificial defects** from notebook 6 (reused verbatim via
  `noisy_engine.jl`) as the correction layer;
* the **non-Clifford circuit runner** (transversal Cliffords + magic-state T/S teleportation), so we can
  watch a real `H·T·CNOT` degrade under noise;
* a small **Monte-Carlo** driver that estimates the logical error rate `p_L` and shows it rise with `p`,
  with `R`, and fall when artificial defects are switched on.

We start from the honest physics question — *shouldn't we integrate a master equation?* — explain why a
stochastic Pauli model is a principled first answer rather than a toy, implement it, and finish with a
concrete roadmap to a fully realistic circuit-level model.


## §1 — From the master equation to a stochastic Pauli model

**The exact picture.** An open quantum system's density matrix obeys a Lindblad master equation

$$\dot\rho = -\tfrac{i}{\hbar}[H,\rho] \;+\; \sum_k \Big(L_k\,\rho\,L_k^\dagger - \tfrac12\{L_k^\dagger L_k,\rho\}\Big),$$

where the jump operators $L_k$ encode relaxation ($T_1$: $L=\sqrt{\Gamma_1}\,\sigma_-$), dephasing
($T_2$: $L=\sqrt{\Gamma_\phi}\,\sigma_z$), leakage, crosstalk, and so on. Integrating this over the
duration of a syndrome round gives a completely-positive trace-preserving (CPTP) map
$\rho \mapsto \mathcal{E}(\rho)=\sum_i K_i\,\rho\,K_i^\dagger$ (a Kraus/channel form). Simulating that
faithfully means propagating a **density matrix** (or averaging many stochastic *quantum trajectories*),
which is far heavier than the pure-state MPS we use — and for a coherent, non-Pauli channel the errors
build up phases that our stabiliser bookkeeping cannot track cheaply.

**Why a stochastic Pauli model is the right first move — and not a toy.** Two facts rescue us:

1. **Pauli twirling.** Averaging any single-qubit channel over the Pauli group turns it into a *Pauli
   channel* $\rho\mapsto (1-p)\rho + p_x X\rho X + p_y Y\rho Y + p_z Z\rho Z$. Amplitude damping twirls
   to a mix of $X$ and $Z$; dephasing twirls to pure $Z$. So "random X/Y/Z at rate p" is exactly the
   twirl of a real hardware channel — the numbers $p_x,p_y,p_z$ are what you'd extract from $T_1,T_2$.
2. **The decoder only sees classical syndromes.** A stabiliser code + MWPM decoder acts on the syndrome
   *bit string*, never on quantum coherences. For a **Clifford** circuit with Pauli noise, the twirl is
   free — it changes nothing the decoder can observe — so the Monte-Carlo logical error rate we compute
   below is the **exact channel-averaged** $p_L$, not an approximation.

So we simulate faithfully by: at each round, **sample** a Pauli error per qubit with probability `p` and
a measurement flip with probability `q`, apply the sampled Paulis to the MPS, and let the sliding-window
decoder do its job. The state stays pure; the randomness lives in *which* faults we draw.

**Where this stops being exact** (and what §10 sketches fixing): coherent over-rotations that don't
commute through later Cliffords, the **T-gates** themselves (the twirl argument assumes Clifford), leakage
out of the qubit subspace, and spatially/temporally *correlated* faults (hook errors from the CNOT
schedule, crosstalk). Those need a density-matrix / trajectory treatment or a circuit-level detector-error
model. But the stochastic Pauli model is the correct, standard foundation everyone builds threshold
studies on — so we build it first.


In [12]:
# Reuse the notebook-6 engine verbatim (MPS, software Pauli frame, sliding-window MWPM with artificial
# defects, transversal gates, magic-state T/S teleportation, exact 2-qubit reference).
include("noisy_engine.jl")
using Printf, Random
Random.seed!(0xC0FFEE)  # reproducible noise draws
println("engine + noise-modelling notebook ready")

patch: 9 data, 4 Z-checks, 4 X-checks   |   MPS sites (3 patches): 51
prep ready: 16 X-recoveries, 16 Z-recoveries (into the frame)
matching graph + subset-DP MWPM loaded
sliding-window epoch decoder loaded
transversal gates + frame/reference propagation loaded
frame-corrected readout (incl. Y) loaded
T-gate commitment loaded
runner ready
exact reference loaded
engine + noise-modelling notebook ready


## §2 — The noise model: rates `p`, `q`, and the timestep `R`

We package the model in a `NoiseModel`:

* **`p`** — probability that a data qubit takes a Pauli error in **one round**. When it fires, the type
  is `X`, `Y` or `Z` with weights `(wx,wy,wz)`; equal weights = the depolarising channel (each with
  probability `p/3`). A `Y` is applied as a coincident `X` and `Z`, matching the engine's convention.
* **`q`** — probability a single stabiliser *measurement* returns the wrong bit that round (a classical
  readout flip; the quantum state is untouched). This is what the decoder's **time edges** exist to catch.
* **`wx,wy,wz`** — the bias. Real superconducting/neutral-atom qubits are strongly **Z-biased**
  (dephasing dominates); §8 explores this.
* **`p_gate`** — an optional extra depolarising burst right after a transversal entangling CNOT, since
  two-qubit gates are the noisiest operation. Leave it `0` to fold gate noise into the idle rounds.

**The timestep.** There is no separate "seconds" variable — *time is counted in syndrome rounds*. An
epoch of `R` rounds is the interval between two logical gates. A qubit idling for `R` rounds is hit with
probability $1-(1-p)^R \approx pR$, so **more rounds between gates ⇒ more accumulated noise** (§6 makes
this quantitative). This is the honest way the "time between gates" enters: you don't get error-for-free
by waiting, and you don't get to measure infinitely often for free either (each round is itself `q`-noisy
and takes time). `sample_epoch_noise` draws one epoch's faults in the exact format the engine's
`run_epoch!` already accepts (`data_errs` / `meas_errs`), so the decoder code is untouched.


In [13]:
"""
    NoiseModel

Phenomenological stochastic Pauli noise model applied per syndrome round.

Fields
- `p`      : probability a given DATA qubit suffers a Pauli error in ONE round. Type drawn from `{X,Y,Z}`
  with weights `wx,wy,wz` (equal = depolarising). A `:Y` = coincident `X` and `Z` on the same qubit.
- `q`      : probability a given stabiliser MEASUREMENT records the wrong bit that round (classical flip;
  the state is untouched) — the fault the decoder's time edges catch.
- `wx,wy,wz` : X/Y/Z branching weights (normalised internally). `wz ≫ wx,wy` = a Z-biased channel.
- `p_gate` : optional extra depolarising prob per data qubit right after a transversal entangling CNOT.

Time is measured in rounds: an `R`-round epoch exposes each qubit to error with prob ≈ `1-(1-p)^R`, so
larger `R` (more time between gates) means more accumulated noise.
"""
struct NoiseModel
    p::Float64; q::Float64; wx::Float64; wy::Float64; wz::Float64; p_gate::Float64
end
NoiseModel(; p=0.0, q=0.0, wx=1.0, wy=1.0, wz=1.0, p_gate=0.0) = NoiseModel(p,q,wx,wy,wz,p_gate)

"""
    draw_pauli(nm) -> Symbol

Sample which Pauli fires GIVEN an error occurred: `:X`, `:Y` or `:Z` with probability proportional to
`wx,wy,wz`. Equal weights give the uniform depolarising branch (each `p/3` unconditionally).

Arguments
- `nm::NoiseModel` : supplies the branching weights `wx,wy,wz`.
"""
function draw_pauli(nm::NoiseModel)
    u = rand() * (nm.wx + nm.wy + nm.wz)
    u < nm.wx ? :X : u < nm.wx + nm.wy ? :Y : :Z
end

"""
    sample_epoch_noise(R, nm; perfect_last_meas) -> (data_errs, meas_errs)

Draw one epoch's stochastic faults in the exact format `run_epoch!` consumes. Each round: every data
qubit fires a Pauli with prob `nm.p`; every check's recorded bit flips with prob `nm.q`. A `:Y` is two
entries (`X` and `Z`) on the same qubit/round.

Arguments
- `R::Int`          : number of syndrome rounds in the epoch (the timestep — see §2).
- `nm::NoiseModel`  : the rates and bias to sample from.
- `perfect_last_meas::Bool` : if true, suppress measurement noise on the final round `R`, giving a clean
  closing time-boundary so logical failure is well defined in a memory experiment (default false).

Returns `data_errs::Vector{(round,"X"/"Z",coord)}`, `meas_errs::Vector{(round,:Z/:X,check_index)}`.
"""
function sample_epoch_noise(R::Int, nm::NoiseModel; perfect_last_meas::Bool=false)
    data_errs = Tuple{Int,String,Tuple{Int,Int}}[]; meas_errs = Tuple{Int,Symbol,Int}[]
    for r in 1:R
        for q in data_coords
            if rand() < nm.p
                P = draw_pauli(nm)
                P === :X ? push!(data_errs,(r,"X",q)) :
                P === :Z ? push!(data_errs,(r,"Z",q)) :
                           (push!(data_errs,(r,"X",q)); push!(data_errs,(r,"Z",q)))
            end
        end
        if !(perfect_last_meas && r == R)
            for i in 1:Nz_per_patch; (rand() < nm.q) && push!(meas_errs,(r,:Z,i)); end
            for i in 1:Nx_per_patch; (rand() < nm.q) && push!(meas_errs,(r,:X,i)); end
        end
    end
    data_errs, meas_errs
end
println("NoiseModel + samplers loaded")

NoiseModel + samplers loaded


## §3 — Reading out logical failure: an exact per-shot oracle

To score a run we need a clean definition of "did a logical error occur?". We use the frame-corrected
logical expectation

`logical_readout(M,p,L) = joint_expect(ψ,[(p,L)]) · frame_sign(M,p,L)`,

i.e. the **exact** (non-collapsing) expectation of the logical operator on the physical MPS, times the
sign the decoder's software Pauli frame prescribes. For a logical basis state under Pauli noise this is
exactly **±1**: `+1` if the decoder returned the qubit to the correct logical coset, `−1` if a logical
error slipped through. Crucially the oracle itself carries **no sampling noise** — the only randomness in
`estimate_pL` is the honest shot-to-shot draw of the fault pattern, so a handful of shots already give a
meaningful binomial estimate.

The canonical experiment is a **logical memory**: prepare `|0>_L` (to measure the logical-`X` error rate
via `<Z_L>`) or `|+>_L` (logical-`Z` error rate via `<X_L>`), idle for `R` noisy rounds under the
sliding-window decoder, and check the sign. Total exposure time = `R` rounds, so `R` is our time axis.


In [14]:
"""
    logical_readout(M, p, L) -> Float64

Frame-corrected single-patch logical expectation `<L>`: exact (non-collapsing) expectation of the
logical operator on the physical MPS times the Pauli-frame sign. For a logical basis state under Pauli
noise this is exactly ±1 — a NOISELESS per-shot pass/fail oracle.

Arguments
- `M::Machine` : the machine whose MPS `M.psi` and software frame `M.fx/M.fz` are read.
- `p::Int`     : which patch (1, 2 or the ancilla 3) to read out.
- `L::Symbol`  : logical basis, `:Z` or `:X`.
"""
logical_readout(M, p, L) = joint_expect(M.psi, [(p, L)]) * frame_sign(M, p, L)

"""
    run_memory_experiment(sym, R, nm; C, B, use_AD, cutoff) -> Machine

Prepare patch 1 in logical state `sym`, expose it to `R` noisy rounds (final round measurement-clean),
sliding-window decode into the software frame, and return the machine. Total exposure time = `R` rounds.

Arguments
- `sym::Symbol`     : logical state to prepare (`:zero`, `:plus`, ...; see the engine's `seed_ops`).
- `R::Int`          : number of noisy syndrome rounds (the exposure time).
- `nm::NoiseModel`  : the noise to sample.
- `C, B` (kw)       : sliding-window commit-core size and look-ahead buffer size, forwarded to
  `run_epoch!` (window `W = C+B`; see notebook 6).
- `use_AD::Bool` (kw) : deposit artificial defects at window commit surfaces (default true).
- `cutoff` (kw)     : MPS bond-truncation cutoff (default `threshold`).
"""
function run_memory_experiment(sym::Symbol, R::Int, nm::NoiseModel; C=2, B=2, use_AD=true, cutoff=threshold)
    M = Machine(MPS(sites, "Up")); prepare_logical!(M, 1, sym; cutoff)
    de, me = sample_epoch_noise(R, nm; perfect_last_meas=true)
    run_epoch!(M, 1, R; data_errs=de, meas_errs=me, C, B, use_AD, cutoff)
    M
end

"""
    estimate_pL(sym, L, R, nm, N; C, B, use_AD, seed) -> (pL, se)

Monte-Carlo logical error probability of the `R`-round memory of `sym`, read in basis `L`, under `nm`.
A shot fails iff `logical_readout < 0`. Returns the failure fraction and its binomial std error
`sqrt(pL(1-pL)/N)`. All variance is honest sampling of the random fault pattern (exact oracle).

Arguments
- `sym::Symbol`     : logical state prepared each shot (e.g. `:zero` for a Z-memory, `:plus` for X).
- `L::Symbol`       : readout basis `:Z` or `:X` (pair it with `sym`: `:zero`↔`:Z`, `:plus`↔`:X`).
- `R::Int`          : rounds per shot (exposure time).
- `nm::NoiseModel`  : the noise to sample.
- `N::Int`          : number of Monte-Carlo shots.
- `C, B, use_AD` (kw) : sliding-window commit-core size, buffer size, and artificial-defect toggle,
  forwarded to `run_memory_experiment` (see notebook 6).
- `seed` (kw)       : if not `nothing`, re-seed the RNG before the loop for reproducibility (default `nothing`).
"""
function estimate_pL(sym, L, R, nm, N; C=2, B=2, use_AD=true, seed=nothing)
    seed !== nothing && Random.seed!(seed)
    fails = 0
    for _ in 1:N
        M = run_memory_experiment(sym, R, nm; C, B, use_AD)
        (logical_readout(M, 1, L) < 0) && (fails += 1)
    end
    pL = fails / N; (pL, sqrt(pL*(1-pL)/N))
end
println("logical-failure oracle + Monte-Carlo estimator loaded")

logical-failure oracle + Monte-Carlo estimator loaded


## §4 — Anatomy of a single noisy shot

Before any statistics, let's watch one shot end-to-end: how many physical faults were drawn, how many
detection events they lit, what net correction the sliding-window decoder committed to the frame, and
whether the logical survived.


In [15]:
"""
    anatomy(R, nm; seed) -> nothing

Draw and print one noisy `R`-round Z-memory shot: the number of sampled data/measurement faults, the
weight of the committed frame (`|fx|`,`|fz|`), and the pass/fail verdict. A window onto one MC sample.

Arguments
- `R::Int`         : rounds in the shot.
- `nm::NoiseModel` : the noise to sample.
- `seed` (kw)      : RNG seed so the printed shot is reproducible (default 7).
"""
function anatomy(R, nm; seed=7)
    Random.seed!(seed)
    de, me = sample_epoch_noise(R, nm; perfect_last_meas=true)
    M = Machine(MPS(sites, "Up")); prepare_logical!(M, 1, :zero)
    run_epoch!(M, 1, R; data_errs=de, meas_errs=me, C=2, B=2, use_AD=true)
    fw = sum(M.fx[1]); fzw = sum(M.fz[1])
    val = logical_readout(M, 1, :Z)
    @printf("R=%d rounds, p=%.3f q=%.3f\n", R, nm.p, nm.q)
    @printf("  sampled faults : %d data-Pauli entries, %d measurement flips\n", length(de), length(me))
    @printf("  committed frame: |fx|=%d  |fz|=%d  (decoder's inferred X/Z corrections)\n", fw, fzw)
    @printf("  <Z_L> (frame-corrected) = %+.0f  ->  %s\n", val, val > 0 ? "LOGICAL OK" : "LOGICAL ERROR")
end
anatomy(6, NoiseModel(; p=0.04, q=0.04); seed=11)

R=6 rounds, p=0.040 q=0.040
  sampled faults : 5 data-Pauli entries, 2 measurement flips
  committed frame: |fx|=2  |fz|=2  (decoder's inferred X/Z corrections)
  <Z_L> (frame-corrected) = +1  ->  LOGICAL OK


## §5 — Sanity checks

Two things must hold before the sweeps mean anything: with **no noise** the memory never fails, and with
noise the failure rate is a sensible small number that **grows with `p`**.


In [16]:
println("clean memory (p=q=0), 16 shots each basis — must be exactly 0:")
pz,_ = estimate_pL(:zero, :Z, 6, NoiseModel(), 16; seed=1)
px,_ = estimate_pL(:plus, :X, 6, NoiseModel(), 16; seed=2)
@printf("  Z-memory pL = %.3f   X-memory pL = %.3f\n\n", pz, px)

println("failure grows with p (R=6, N=40, Z-memory):")
for p in (0.01, 0.03, 0.06)
    pl, se = estimate_pL(:zero, :Z, 6, NoiseModel(; p=p, q=p), 40; seed=100)
    @printf("  p=q=%.3f   pL = %.3f ± %.3f\n", p, pl, se)
end

clean memory (p=q=0), 16 shots each basis — must be exactly 0:
  Z-memory pL = 0.000   X-memory pL = 0.000

failure grows with p (R=6, N=40, Z-memory):
  p=q=0.010   pL = 0.025 ± 0.025
  p=q=0.030   pL = 0.025 ± 0.025
  p=q=0.060   pL = 0.325 ± 0.074


## §6 — The timestep study: time between gates

Here is the point the question raised directly: *more syndrome rounds between gates increases the time
between gates, which increases the chance of an error.* We hold the physical rates fixed and sweep the
epoch length `R` (the number of rounds a logical qubit idles between operations). Because each extra
round is another independent chance for every qubit to fault, `p_L` rises with `R` — roughly linearly
while `p_L` is small (each round contributes a near-independent shot at a logical fault). This is the
quantitative face of "waiting longer costs you". It is also why a real machine wants gates to be *fast in
units of the syndrome-round time*, and why you can't drive `p_L` to zero just by measuring more often —
every round you add is itself `q`-noisy and takes wall-clock time.


In [17]:
"""
    ascii_bar(label, value, se, vmax; width) -> nothing

Print one horizontal bar `label | ####  value±se`. A dependency-free stand-in for a plot (this project
pins only ITensors/ITensorMPS).

Arguments
- `label`        : the row label (e.g. `"R=6"` or `"p=0.010"`).
- `value`        : the quantity plotted (bar length).
- `se`           : its standard error, printed after the value.
- `vmax`         : the value that fills the full bar width (the axis maximum).
- `width` (kw)   : bar width in characters (default 40).
"""
function ascii_bar(label, value, se, vmax; width=40)
    n = vmax > 0 ? round(Int, width*value/vmax) : 0
    @printf("  %-10s |%s%s  %.3f ± %.3f\n", label, "#"^n, " "^(width-n), value, se)
end

println("Z-memory pL vs epoch length R  (p=q=0.03, N=40):")
nm = NoiseModel(; p=0.03, q=0.03); Rs = (2,4,6,8,10); res = Tuple{Int,Float64,Float64}[]
for R in Rs
    pl, se = estimate_pL(:zero, :Z, R, nm, 40; seed=200)
    push!(res, (R, pl, se))
end
vmax = maximum(r[2] for r in res) + 1e-6
for (R, pl, se) in res; ascii_bar("R=$R", pl, se, vmax); end
println("\n(more rounds between gates => more accumulated noise => larger pL)")

Z-memory pL vs epoch length R  (p=q=0.03, N=40):
  R=2        |##########                                0.050 ± 0.034
  R=4        |##########                                0.050 ± 0.034
  R=6        |##########                                0.050 ± 0.034
  R=8        |########################################  0.200 ± 0.063
  R=10       |#########################                 0.125 ± 0.052

(more rounds between gates => more accumulated noise => larger pL)


## §7 — Logical error rate vs physical error rate (a threshold-style sweep)

The headline plot of any QEC study: sweep the physical rate `p` and watch the logical rate `p_L`. Below
threshold, a bigger code suppresses `p_L`; above threshold it makes things worse. Our engine is fixed at
distance **d = 3**, so we can't cross curves for two distances here (that needs the distance-parametric
engine sketched in §10), but we *can* show the sub-threshold trend and — reusing the notebook-6 result
directly — that **artificial defects** at the window commit surfaces measurably lower `p_L`.


In [18]:
println("Z-memory pL vs physical rate p  (R=6, N=60, depolarising, q=p):")
ps = (0.005, 0.01, 0.02, 0.04, 0.08); rows = Tuple{Float64,Float64,Float64}[]
for p in ps
    pl, se = estimate_pL(:zero, :Z, 6, NoiseModel(; p=p, q=p), 60; seed=300)
    push!(rows, (p, pl, se))
end
vmax = maximum(r[2] for r in rows) + 1e-6
for (p, pl, se) in rows; ascii_bar(@sprintf("p=%.3f",p), pl, se, vmax); end

println("\nartificial defects on vs off  (p=q=0.04, R=6, N=60):")
for ad in (false, true)
    pl, se = estimate_pL(:zero, :Z, 6, NoiseModel(; p=0.04, q=0.04), 60; use_AD=ad, seed=400)
    @printf("  use_AD=%-5s  pL = %.3f ± %.3f\n", ad, pl, se)
end

Z-memory pL vs physical rate p  (R=6, N=60, depolarising, q=p):
  p=0.005    |                                          0.000 ± 0.000
  p=0.010    |##                                        0.017 ± 0.017
  p=0.020    |###                                       0.033 ± 0.023
  p=0.040    |###############                           0.150 ± 0.046
  p=0.080    |########################################  0.400 ± 0.063

artificial defects on vs off  (p=q=0.04, R=6, N=60):
  use_AD=false  pL = 0.217 ± 0.053
  use_AD=true   pL = 0.200 ± 0.052


## §8 — Biased noise

Real qubits are rarely depolarising. Superconducting and neutral-atom qubits dephase far faster than they
relax, so `Z` errors dominate — a **bias** $\eta = w_z/(w_x+w_y)$ that can reach 100–1000. Bias matters
because a `Z`-memory (logical-`X` error rate, our `<Z_L>` readout) is threatened only by `X` and `Y`
faults, so raising the `Z` weight at fixed total `p` *lowers* the `Z`-memory failure rate — the flip side
is that the `X`-memory suffers. Biased-noise codes (XZZX, tailored surface codes) exploit exactly this.
Here we just show the model responds correctly to the knob.


In [ ]:
println("effect of Z-bias at fixed total p=0.05, q=0.02, R=6, N=48:")
for (name, nm) in (
        ("depolarising", NoiseModel(; p=0.05, q=0.02)),
        ("Z-biased η=10", NoiseModel(; p=0.05, q=0.02, wx=1, wy=1, wz=20)),
        ("Z-biased η=50", NoiseModel(; p=0.05, q=0.02, wx=1, wy=1, wz=100)))
    plz, sez = estimate_pL(:zero, :Z, 6, nm, 48; seed=500)  # threatened by X,Y
    plx, sex = estimate_pL(:plus, :X, 6, nm, 48; seed=501)  # threatened by Z
    @printf("  %-14s  Z-mem pL=%.3f±%.3f   X-mem pL=%.3f±%.3f\n", name, plz, sez, plx, sex)
end
println("\n(more Z-bias => fewer X/Y faults => lower Z-memory pL, higher X-memory pL)")

## §9 — Noise on the full non-Clifford runner

Finally we drive the **complete stack** — transversal Cliffords plus magic-state `T`/`S` teleportation,
with the ancilla commit epoch itself made noisy — through `run_noisy_circuit`. Every logical gate is
followed by a noisy `R`-round idle epoch on both data patches; `p_gate` optionally adds a burst after
each entangling CNOT. We compare the frame-corrected `<ZZ>`/`<XX>` against the exact statevector.

For the **magic Bell** state `H·T·CNOT`, the ideal correlators are `<ZZ>=1` and `<XX>=cos(π/4)=0.707`.
At low `p` the decoder recovers them; as `p` rises they wash out — the visible onset of logical failure
on a genuinely non-Clifford circuit. (Note the caveat from §1: for the non-Clifford `T` the stochastic
Pauli model is an approximation to the true channel, not provably exact — but it is the standard and
useful one.)


In [21]:
"""
    inject_gate_noise!(M, p, nm; cutoff) -> M

Apply one noise layer (each data qubit fires a Pauli with prob `nm.p_gate`, type via `draw_pauli`)
directly to patch `p` of the MPS — the elevated error burst of a transversal entangling gate, caught by
the next epoch's syndrome round.

Arguments
- `M::Machine`     : the machine whose MPS is perturbed.
- `p::Int`         : the patch to hit.
- `nm::NoiseModel` : supplies `p_gate` (the per-qubit burst probability) and the X/Y/Z weights.
- `cutoff` (kw)    : MPS bond-truncation cutoff (default `threshold`).
"""
function inject_gate_noise!(M, p, nm; cutoff=threshold)
    for q in data_coords
        if rand() < nm.p_gate
            P = draw_pauli(nm)
            P === :X ? (M.psi = apply(op("X", site_of[(p,q)]), M.psi; cutoff)) :
            P === :Z ? (M.psi = apply(op("Z", site_of[(p,q)]), M.psi; cutoff)) :
                       (M.psi = apply(op("X", site_of[(p,q)]), M.psi; cutoff);
                        M.psi = apply(op("Z", site_of[(p,q)]), M.psi; cutoff))
        end
    end
    M
end

"""
    teleport_S_noisy!(M, p, nm, R; C, B, use_AD, cutoff, force_wrong) -> M
    teleport_T_noisy!(M, p, nm, R; C, B, use_AD, cutoff, force_wrong) -> M

Noisy variants of the engine's magic-state teleportation: the ancilla commit epoch (`run_epoch!` on
patch 3) is driven with sampled Pauli+measurement noise, so the gadget itself is faulty. Structure and
byproduct logic are identical to the noiseless engine versions.

Arguments (identical for S and T)
- `M::Machine`     : the machine; patch 3 is used as the recyclable magic ancilla.
- `p::Int`         : the data patch receiving the teleported gate.
- `nm::NoiseModel` : noise sampled for the commit epoch.
- `R::Int`         : commit-epoch length in rounds.
- `C, B, use_AD` (kw) : sliding-window commit-core size, buffer size, artificial-defect toggle, forwarded
  to `run_epoch!` (see notebook 6).
- `cutoff` (kw)    : MPS bond-truncation cutoff (default `threshold`).
- `force_wrong::Bool` (kw) : flip the committed teleportation bit to demonstrate an uncorrectable error
  (default false).
"""
function teleport_S_noisy!(M, p, nm, R; C=2, B=2, use_AD=true, cutoff=threshold, force_wrong=false)
    prepare_logical!(M, 3, :Y; cutoff); logical_CNOT!(M, p, 3; cutoff)
    de, me = sample_epoch_noise(R, nm); run_epoch!(M, 3, R; data_errs=de, meas_errs=me, C, B, use_AD, cutoff)
    m_raw = measure_patch_Z_raw!(M, 3; cutoff)
    fbit = 0; for q in ZL_support; fbit ⊻= M.fx[3][didx[q]]; end
    m = m_raw ⊻ fbit ⊻ (force_wrong ? 1 : 0); m == 1 && apply_ZL_phys!(M, p; cutoff); M
end
function teleport_T_noisy!(M, p, nm, R; C=2, B=2, use_AD=true, cutoff=threshold, force_wrong=false)
    prepare_logical!(M, 3, :A; cutoff); logical_CNOT!(M, p, 3; cutoff)
    de, me = sample_epoch_noise(R, nm); run_epoch!(M, 3, R; data_errs=de, meas_errs=me, C, B, use_AD, cutoff)
    m_raw = measure_patch_Z_raw!(M, 3; cutoff)
    fbit = 0; for q in ZL_support; fbit ⊻= M.fx[3][didx[q]]; end
    m = m_raw ⊻ fbit ⊻ (force_wrong ? 1 : 0); m == 1 && teleport_S_noisy!(M, p, nm, R; C, B, use_AD, cutoff); M
end

"""
    apply_noisy_logical!(M, gate, nm, R; C, B, use_AD, cutoff) -> M

Dispatch one logical gate under noise: S/T route to the noisy teleportation gadgets; the transversal
Cliffords (X,Z,H,CNOT) are applied by the engine as instantaneous layers (exposure lives in the
following idle epoch).

Arguments
- `M::Machine`     : the machine acted on.
- `gate::Tuple`    : `(:X|:Z|:H|:S|:T, patch)` or `(:CNOT, ctrl, tgt)`.
- `nm::NoiseModel` : noise for the S/T commit epochs (Cliffords add no noise here themselves).
- `R::Int`         : commit-epoch length passed to the teleportation gadgets.
- `C, B, use_AD, cutoff` (kw) : sliding-window / MPS settings forwarded to the teleport gadgets and
  engine (commit-core size, buffer size, artificial-defect toggle, truncation cutoff).
"""
function apply_noisy_logical!(M, gate, nm, R; C=2, B=2, use_AD=true, cutoff=threshold)
    g = gate[1]
    g === :S ? teleport_S_noisy!(M, gate[2], nm, R; C, B, use_AD, cutoff) :
    g === :T ? teleport_T_noisy!(M, gate[2], nm, R; C, B, use_AD, cutoff) :
               apply_logical!(M, gate, R; C, B, use_AD, cutoff)
end

"""
    run_noisy_circuit(circuit, nm; R, C, B, use_AD, cutoff) -> Machine

Run a full 2-qubit Clifford+T circuit on |0>_L ⊗ |0>_L with stochastic noise everywhere: each gate is
followed by a noisy `R`-round idle epoch on BOTH data patches, T/S use the noisy gadget, and (if
`nm.p_gate>0`) a burst follows each CNOT. Read out with `corr`/`corr2`.

Arguments
- `circuit`        : list of gate tuples, `(:X|:Z|:H|:S|:T, patch)` / `(:CNOT, ctrl, tgt)`.
- `nm::NoiseModel` : the noise applied throughout (idle epochs, T/S commit epochs, and CNOT bursts).
- `R::Int` (kw)    : rounds per idle/commit epoch = the time between gates (default 6).
- `C, B` (kw)      : sliding-window commit-core size and look-ahead buffer size (window `W=C+B`; default 2,2).
- `use_AD::Bool` (kw) : deposit artificial defects at window commit surfaces (default true).
- `cutoff` (kw)    : MPS bond-truncation cutoff (default `threshold`).
"""
function run_noisy_circuit(circuit, nm; R=6, C=2, B=2, use_AD=true, cutoff=threshold)
    M = Machine(MPS(sites, "Up"))
    prepare_logical!(M, 1, :zero; cutoff); prepare_logical!(M, 2, :zero; cutoff)
    for gate in circuit
        apply_noisy_logical!(M, gate, nm, R; C, B, use_AD, cutoff)
        (nm.p_gate > 0 && gate[1] === :CNOT) &&
            (inject_gate_noise!(M, gate[2], nm; cutoff); inject_gate_noise!(M, gate[3], nm; cutoff))
        for pp in (1,2)
            de, me = sample_epoch_noise(R, nm)
            run_epoch!(M, pp, R; data_errs=de, meas_errs=me, C, B, use_AD, cutoff)
        end
    end
    M
end
println("noisy non-Clifford runner loaded")

noisy non-Clifford runner loaded


In [ ]:
circ = [(:H,1),(:T,1),(:CNOT,1,2)]           # magic Bell: ideal <ZZ>=1, <XX>=0.707
ψ = ref_state(circ); rZZ = ref2(ψ,:Z,:Z); rXX = ref2(ψ,:X,:X)
println("noisy magic Bell (H1;T1;CNOT), R=4 — frame-corrected <ZZ>,<XX> vs exact ref:")
@printf("  ideal:            ZZ=%.3f  XX=%.3f\n", rZZ, rXX)
for p in (0.0, 0.01, 0.03, 0.08, 0.09, 0.1, 0.11, 0.12, 0.13, 0.14, 0.15)
    Random.seed!(600)
    M = run_noisy_circuit(circ, NoiseModel(; p=p, q=p); R=4)
    @printf("  p=q=%.3f:  ZZ=%+.3f  XX=%+.3f\n", p, corr2(M,:Z,:Z), corr2(M,:X,:X))
end
println("\n(low p: recovered; growing p: correlators wash out = onset of logical failure)")

noisy magic Bell (H1;T1;CNOT), R=4 — frame-corrected <ZZ>,<XX> vs exact ref:
  ideal:            ZZ=1.000  XX=0.707
  p=q=0.000:  ZZ=+1.000  XX=+0.707
  p=q=0.010:  ZZ=+1.000  XX=+0.707
  p=q=0.030:  ZZ=+1.000  XX=+0.707
  p=q=0.080:  ZZ=+1.000  XX=+0.707
  p=q=0.090:  ZZ=+1.000  XX=-0.707
  p=q=0.100:  ZZ=+1.000  XX=-0.707
  p=q=0.110:  ZZ=-1.000  XX=-0.707
  p=q=0.120:  ZZ=+1.000  XX=+0.707
  p=q=0.130:  ZZ=+1.000  XX=-0.707
  p=q=0.140:  ZZ=-1.000  XX=-0.707
  p=q=0.150:  ZZ=-1.000  XX=-0.707

(low p: recovered; growing p: correlators wash out = onset of logical failure)


: 

## §10 — From this model to a realistic one

The model above is the *phenomenological* rung of the ladder — the standard first step, exact for
Clifford + Pauli noise. Here is the road to a fully realistic simulation, roughly in order of payoff:

1. **Circuit-level noise (detector-error model).** Attach a fault channel to *every physical operation*
   inside a round — after each CNOT in the stabiliser-extraction schedule, after resets, before each
   measurement — not just to the idle data qubits. This produces **hook errors**: a single CNOT fault
   spreads to two data qubits, creating space-time-correlated detector pairs that don't lie on a simple
   graph edge. The matching graph then needs edges weighted by $-\log(p_\text{edge})$ from a compiled
   **detector-error model**, and diagonal/hook edges added. This is the single most important step toward
   a real threshold number (and drops the threshold from the phenomenological ~3% toward ~1%).

2. **A distance sweep (d = 3 vs d = 5).** The one piece of physics this notebook can't show: below
   threshold, $p_L \sim (p/p_\text{th})^{\lfloor d/2\rfloor+1}$, so curves for different `d` **cross** at
   $p_\text{th}$. The engine hard-codes `const d = 3`; making the geometry (`data_coords`, the check
   lists, `XREC`/`ZREC`, the matching graph) a function of `d` and re-running the §7 sweep for d=3,5,7 is
   the natural next notebook — the deferred "threshold sweep" task.

3. **Weighted / approximate matching.** Our exact subset-DP MWPM is $O(2^n n)$ — fine for d=3, hopeless
   at d=5+ under realistic densities. Swap in a real backend (Blossom V, PyMatching, or union–find) with
   the DEM edge weights. This also lets edges carry probabilities, so the decoder uses the *bias* of §8.

4. **Beyond Pauli — the master equation, honestly.** Replace the sampled Pauli channel with the actual
   CPTP maps: **amplitude damping** ($T_1$) and **dephasing** ($T_2$) with real time constants per
   physical operation, giving `p ≈ t_round/T_1 + ...`. Coherent errors (systematic over/under-rotations)
   need either a density-matrix simulation, a **quantum-trajectory** unravelling averaged over many
   shots, or a Pauli-twirl approximation whose error you bound separately. Our pure-state MPS supports
   trajectories directly (each shot is one trajectory); density matrices would double the site count.

5. **Leakage and correlated faults.** Real qubits leak out of the computational subspace (needs qutrit
   sites and leakage-reduction circuits) and suffer **crosstalk** (a gate on one qubit dephases its
   neighbours) — spatially correlated noise the i.i.d. model omits. These degrade the threshold and are
   what leakage-reduction units and dynamical decoupling are designed to fight.

6. **Gate- and measurement-dependent rates.** Give two-qubit gates, single-qubit gates, idle, reset and
   measurement each their own error rate (hardware calibration data), instead of one global `p`. `p_gate`
   in §9 is the first token of this; a full model has a small dictionary of per-operation rates.

7. **The lattice-surgery seam as a temporal slab.** For multi-patch logical gates done by lattice surgery
   (rather than the transversal CNOT used here), the merge outcome is a *temporal* observable measured
   over `d` rounds — the **slab** decoder of `6.Real-Time-Decoder/example_sliding_window_slab.ipynb`.
   Under this stochastic noise model, a single seam measurement error is outvoted by the slab, quadratic
   in `q` rather than linear — worth folding in once lattice-surgery gates enter the runner.

Items 1–3 turn this teaching model into a quantitative threshold experiment; 4–5 turn it into a hardware
model. The abstraction we deliberately kept is that faults are **sampled and applied as Paulis to a pure
MPS** — which, per §1, is exactly right for the Clifford backbone and a controlled approximation for the
magic-state `T`.


## §11 — What this notebook establishes

* A **stochastic Pauli + measurement noise model** (`NoiseModel`, `sample_epoch_noise`) that fires X/Y/Z
  data errors at rate `p` and measurement flips at rate `q` on **every** syndrome round, driving the
  notebook-6 sliding-window MWPM decoder through its existing injection hooks — no decoder code changed.
* An explicit **timestep**: `R` rounds per epoch is the time between gates, and §6 shows `p_L` rising
  with `R` — the honest cost of idling — while §7 shows it rising with `p` and falling with artificial
  defects, and §8 shows the model responding correctly to noise **bias**.
* A **noiseless per-shot logical-failure oracle** (`logical_readout`, exact ±1) feeding a small
  **Monte-Carlo** estimator of `p_L`, applied both to logical memories and to the **full non-Clifford
  runner** (`run_noisy_circuit`) with a noisy magic-state gadget.
* A grounded account of **why** a stochastic Pauli model is the principled first rung (Pauli twirling +
  the decoder only sees classical syndromes) and a concrete **roadmap** (circuit-level DEM, distance
  sweep, weighted matching, master-equation/trajectory noise, leakage, per-operation rates, the LS slab)
  to a fully realistic simulation.
